# Phase C — planar (xy/yz/zx) PL surrogate (M4)

Three 2-D UNets, one per axis-aligned **slice stack**, replace the data-hungry
full-3-D UNet. Each net predicts a 2-D PL plane from `(material slice, Tx, freq)`;
stacking a net's slices rebuilds a volume, and the three volumes are **composed**
(mean, then an optional learned fusion). PL only — arrival-time is not sliceable
and the browser reads it from the cache/analytic tiers.

**Reuses the existing PL volumes** (`dataset/shard_*_pl.npy`) by slicing them, so no
regeneration is needed beyond running `phase_b3_dataset.ipynb` PL-only once.
Featurization lives in `dataset_3d.py` (`plane_input`, `slice_plane`, `stack_planes`)
so the browser can mirror one reference. Port of the proven 2-D recipe
(`Physics Engine/2D/SIM V2/phase_c_v2_train.ipynb`): masked-MSE + gradient-L1,
AdamW→cosine, AMP, early stop, ONNX parity gate.

In [ ]:
#@title Run mode  (re-run if you change it; other cells read these)
RUN_MODE      = "full"   #@param ["full", "smoke"]
TRAIN_ORIENTS = ["zx", "xy", "yz"]   #@param  (zx = horizontal floors, the workhorse)
BASE          = 64       #@param {type:"integer"}
LR            = 1e-3     #@param {type:"number"}
BS            = 16       #@param {type:"integer"}
EPOCHS        = 120      #@param {type:"integer"}
PATIENCE      = 15       #@param {type:"integer"}
SLICES_PER_TX = 8        #@param {type:"integer"}   training slices sampled per (Tx,band)
POOL_THRESH   = 24       #@param {type:"integer"}   pool an axis only if it is >= this
N_TEST        = 24       #@param {type:"integer"}   test Tx used for volume reconstruction
NW            = 2        #@param {type:"integer"}
SEED          = 0

if RUN_MODE == "smoke":
    EPOCHS, SLICES_PER_TX, N_TEST, NW = 2, 3, 4, 0
print(f"RUN_MODE={RUN_MODE}  orients={TRAIN_ORIENTS}  base={BASE}  epochs={EPOCHS}")

In [ ]:
#@title Setup: Drive, device, scene assets (no engine needed — we slice stored PL)
import os, sys, json, time, math, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = "/content/drive/MyDrive/indoor-walk-test-main/Physics Engine/3D Map Physics/SIM V1 3D"  #@param {type:"string"}
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception:
    pass
ROOT = str(__import__("pathlib").Path(ROOT).expanduser().resolve())
sys.path.insert(0, ROOT)
import dataset_3d as D

torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
dev = "cuda" if torch.cuda.is_available() else "cpu"

man    = json.load(open(f"{ROOT}/manifest_3d.json"))
M      = np.load(f"{ROOT}/material_grid.npy")
inside = np.load(f"{ROOT}/inside_mask.npy")
norm   = D.load_norm(man)
CELL   = float(man["cell_size_m"])
PL_LO, PL_RNG = norm.pl_min_db, norm.pl_range_db
DATA   = f"{ROOT}/dataset"
CKPT_D = f"{ROOT}/checkpoints"; os.makedirs(CKPT_D, exist_ok=True)
WEB    = f"{ROOT}/web";         os.makedirs(WEB, exist_ok=True)

print("device  ", dev, "| torch", torch.__version__)
print("grid    ", M.shape, "| interior", f"{int(inside.sum()):,}")
print("scene_sha", D.scene_sha(M), "| plane channels", len(D.INPUT_CHANNELS_PLANE))
for o in D.PLANE_ORIENTS:
    print(f"  {o}: plane {D.plane_shape(M.shape, o)}  x {D.n_slices(M.shape, o)} slices")

## Dataset — one `PlaneDS` per orientation

A training item is `(Tx, band, orientation, slice)`; the target is that slice of the
stored PL volume, the input is `plane_input(...)` (10 channels: 6 material one-hot of
the slice + Tx blob + freq + log-distance + perpendicular offset). Off-Tx slices are
included so each stack learns to predict every slice it must contribute to a volume.

In [ ]:
#@title PlaneDS
sp = json.load(open(f"{DATA}/splits.json"))
assert sp.get("scene_sha", D.scene_sha(M)) == D.scene_sha(M), "splits.json is a different scene"

def _spread(valid, tx_fixed, k):
    """`k` slices spread over the interior-bearing slices, always incl. the Tx's own."""
    if len(valid) <= k:
        return list(valid)
    pick = np.linspace(0, len(valid) - 1, k).round().astype(int)
    out = {int(valid[p]) for p in pick}
    out.add(int(valid[np.argmin(np.abs(np.asarray(valid) - tx_fixed))]))
    return sorted(out)

class PlaneDS(Dataset):
    def __init__(self, data_dir, keep_pos, orient, *, full=False, slices_per_tx=SLICES_PER_TX):
        self.orient, self.full = orient, full
        self.fa = D._ORIENT_FIXED_AXIS[orient]
        nfix = D.n_slices(M.shape, orient)
        self.valid = [k for k in range(nfix) if D.slice_plane(inside, orient, k).any()]
        keep = set(int(p) for p in keep_pos)
        self.pl, self.items = [], []
        for mp in D.list_shards(data_dir):
            s = int(os.path.basename(mp).split("_")[1])
            pl, _tau, meta = D.open_shard(data_dir, s)          # tau is None (PL-only)
            si = len(self.pl); self.pl.append(pl)
            for i, pid in enumerate(meta["pos_id"]):
                if int(pid) not in keep:
                    continue
                tx = tuple(int(v) for v in meta["tx"][i])
                ff = float(meta["freq_feat"][i])
                idxs = self.valid if full else _spread(self.valid, tx[self.fa], slices_per_tx)
                self.items += [(si, i, tx, ff, int(k)) for k in idxs]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, j):
        si, i, tx, ff, k = self.items[j]
        vol = np.asarray(self.pl[si][i], np.float32)            # (X,Y,Z) normalized PL
        y = D.slice_plane(vol, self.orient, k)                  # (H,W)
        x = D.plane_input(M, tx, ff, self.orient, k, CELL)      # (10,H,W)
        m = D.slice_plane(inside, self.orient, k).astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y[None]), torch.from_numpy(m[None])

def loaders(orient):
    tr = PlaneDS(DATA, sp["train"], orient)
    va = PlaneDS(DATA, sp["val"], orient)
    if len(PlaneDS(DATA, sp["train"], orient, full=False).items) == 0:
        raise RuntimeError("no training planes — run phase_b3_dataset.ipynb (PL-only) first")
    tl = DataLoader(tr, BS, shuffle=True, num_workers=NW, pin_memory=(dev == "cuda"))
    vl = DataLoader(va, BS, shuffle=False, num_workers=NW, pin_memory=(dev == "cuda"))
    return tr, va, tl, vl

for o in TRAIN_ORIENTS:
    tr = PlaneDS(DATA, sp["train"], o); va = PlaneDS(DATA, sp["val"], o)
    print(f"{o}: train {len(tr)} planes  val {len(va)}  ({len(tr.valid)} interior slices)")

## 2-D UNet + loss

Base-64 UNet mirroring the 3-D one but 2-D, with **anisotropic pooling**: an axis is
halved only if it is `>= POOL_THRESH`, so the thin 17-voxel vertical axis of the `xy`
and `yz` stacks is never collapsed. Objective is the proven 2-D one — masked-MSE +
0.1·gradient-L1 — with the mask being the plane's interior slice.

In [ ]:
#@title UNet2D, pooling, loss
def dconv(ci, co):
    return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True),
                         nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True))

class UNet2D(nn.Module):
    def __init__(self, cin, base=BASE, pool=(2, 2)):
        super().__init__()
        self.pool = nn.MaxPool2d(pool)
        self.e1 = dconv(cin, base); self.e2 = dconv(base, base * 2)
        self.b = dconv(base * 2, base * 4)
        self.d2 = dconv(base * 4 + base * 2, base * 2); self.d1 = dconv(base * 2 + base, base)
        self.out = nn.Conv2d(base, 1, 1)

    def _up(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return torch.cat([x, skip], 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); b = self.b(self.pool(e2))
        d2 = self.d2(self._up(b, e2)); d1 = self.d1(self._up(d2, e1))
        return torch.sigmoid(self.out(d1))            # normalized PL in [0,1]

def orient_pool(orient):
    H, W = D.plane_shape(M.shape, orient)
    return (2 if H >= POOL_THRESH else 1, 2 if W >= POOL_THRESH else 1)

def grad_l1(a, b):
    return (((a[:, :, 1:, :] - a[:, :, :-1, :]) - (b[:, :, 1:, :] - b[:, :, :-1, :])).abs().mean()
            + ((a[:, :, :, 1:] - a[:, :, :, :-1]) - (b[:, :, :, 1:] - b[:, :, :, :-1])).abs().mean())

def masked_mse(p, y, m):
    return (((p - y) ** 2) * m).sum() / m.sum().clamp(min=1)

GRAD_W = 0.1
def total_loss(p, y, m):
    return masked_mse(p, y, m) + GRAD_W * grad_l1(p * m, y * m)

for o in TRAIN_ORIENTS:
    print(f"{o}: pool {orient_pool(o)}  plane {D.plane_shape(M.shape, o)}")

## Train — one net per orientation (resumable, AMP, cosine, early stop)

Checkpoints per orientation to Drive every 2 epochs; a Colab disconnect resumes the
same net. Val RMSE is reported in dB (`PL_RNG·sqrt(masked_mse)`).

In [ ]:
#@title Train all orientations
amp_dtype = torch.bfloat16 if (dev == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

def lr_at(step, total, warm=0.1):
    t = min(step / max(total, 1), 1.0)
    if t < warm:
        return LR * max(t / warm, 1e-3)
    c = (t - warm) / (1 - warm)
    return LR * 0.02 + 0.5 * LR * 0.98 * (1 + math.cos(math.pi * c))

@torch.no_grad()
def val_rmse_db(net, vl):
    net.eval(); se = n = 0.0
    for x, y, m in vl:
        x, y, m = x.to(dev), y.to(dev), m.to(dev)
        with torch.autocast(dev, amp_dtype, enabled=(dev == "cuda")):
            p = net(x).float()
        se += float(((p - y) ** 2 * m).sum()); n += float(m.sum().clamp(min=1))
    return PL_RNG * math.sqrt(se / max(n, 1))

def train_orient(orient):
    tr, va, tl, vl = loaders(orient)
    net = UNet2D(len(D.INPUT_CHANNELS_PLANE), pool=orient_pool(orient)).to(dev)
    opt = torch.optim.AdamW(net.parameters(), LR, weight_decay=1e-4)
    scaler = torch.amp.GradScaler(dev, enabled=(dev == "cuda" and amp_dtype == torch.float16))
    ckpt = f"{CKPT_D}/best_pl_unet2d_{orient}.pt"
    res = f"{CKPT_D}/resume_pl_unet2d_{orient}.pt"
    total = EPOCHS * max(len(tl), 1)
    start, best, bad, g = 0, 1e9, 0, 0
    if os.path.exists(res):
        st = torch.load(res, map_location=dev)
        net.load_state_dict(st["net"]); opt.load_state_dict(st["opt"])
        scaler.load_state_dict(st["scaler"]); start, best, g = st["epoch"] + 1, st["best"], st["g"]
        print(f"[{orient}] resumed at epoch {start} (best {best:.2f} dB)")
    for ep in range(start, EPOCHS):
        net.train(); t0 = time.time()
        for x, y, m in tl:
            x, y, m = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True), m.to(dev, non_blocking=True)
            for pg in opt.param_groups:
                pg["lr"] = lr_at(g, total)
            with torch.autocast(dev, amp_dtype, enabled=(dev == "cuda")):
                loss = total_loss(net(x).float(), y, m)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt); scaler.update(); g += 1
        rmse = val_rmse_db(net, vl)
        if rmse < best - 1e-4:
            best, bad = rmse, 0; torch.save(net.state_dict(), ckpt)
        else:
            bad += 1
        torch.save(dict(net=net.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                        epoch=ep, best=best, g=g), res)
        print(f"[{orient}] ep {ep+1:03d}/{EPOCHS}  val {rmse:5.2f} dB  best {best:5.2f}  ({time.time()-t0:.0f}s)")
        if bad >= PATIENCE:
            print(f"[{orient}] early stop"); break
    net.load_state_dict(torch.load(ckpt, map_location=dev))
    return net, best

models, best_db = {}, {}
for o in TRAIN_ORIENTS:
    models[o], best_db[o] = train_orient(o)
print("\nbest plane RMSE (dB):", {o: round(v, 2) for o, v in best_db.items()})

## Compose to 3-D + validate against the stored volumes

Each net predicts all its interior slices for a test Tx; stacking rebuilds a volume
estimate, and the three are fused (mean-in-dB baseline). The stored PL volume is the
engine's own solve, so RMSE of the fused volume against it is the number that matters
— compare to the full-3-D surrogate's **14.47 dB**.

In [ ]:
#@title Reconstruct + fuse + volume RMSE vs the engine (stored) volumes
@torch.no_grad()
def reconstruct(tx, ff):
    est = {}
    for o, net in models.items():
        net.eval()
        planes = [net(torch.from_numpy(D.plane_input(M, tx, ff, o, k, CELL))[None].to(dev))
                  [0, 0].float().cpu().numpy() for k in range(D.n_slices(M.shape, o))]
        est[o] = D.stack_planes(planes, o)                      # (X,Y,Z) normalized
    fused = np.mean(list(est.values()), 0)
    return est, fused

def collect_test(n_max):
    out = []
    for mp in D.list_shards(DATA):
        s = int(os.path.basename(mp).split("_")[1])
        pl, _t, meta = D.open_shard(DATA, s)
        for i, pid in enumerate(meta["pos_id"]):
            if int(pid) in set(sp["test"]):
                out.append((np.asarray(pl[i], np.float32), tuple(int(v) for v in meta["tx"][i]),
                            float(meta["freq_feat"][i]), float(meta["freq_mhz"][i])))
            if len(out) >= n_max:
                return out
    return out

def rmse_db(a_norm, b_norm, m):
    e = (a_norm - b_norm)[m] * PL_RNG
    return float(np.sqrt((e ** 2).mean()))

test = collect_test(N_TEST)
agg = {o: [] for o in TRAIN_ORIENTS}; agg["fused"] = []; agg["fspl"] = []
for gt, tx, ff, fmhz in test:
    est, fused = reconstruct(tx, ff)
    for o in TRAIN_ORIENTS:
        agg[o].append(rmse_db(est[o], gt, inside))
    agg["fused"].append(rmse_db(fused, gt, inside))
    d = D.distance_m(tx, D.voxel_coords(M.shape), CELL)
    fspl_norm = np.clip((D.fspl_db(d, fmhz, man) - PL_LO) / PL_RNG, 0, 1)
    agg["fspl"].append(rmse_db(fspl_norm, gt, inside))

print(f"volume RMSE over {len(test)} test Tx (dB):")
for k in list(TRAIN_ORIENTS) + ["fused", "fspl"]:
    print(f"  {k:>6}: {np.mean(agg[k]):5.2f}")
print(f"\nfull-3D surrogate baseline was 14.47 dB — fused target <= 5-8 dB")

## Export ONNX (×3) + parity gate + write contracts

One `pl_unet2d_<orient>.onnx` + `pl_unet2d_<orient>.json` per stack. The browser's
consume/compose path is a follow-up (out of scope here); these are what it will read.

In [ ]:
#@title Export + parity + contracts
try:
    import onnxruntime as ort
except ModuleNotFoundError:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "onnx"], check=True)
    import onnxruntime as ort

PARITY_DB = 0.1
for o, net in models.items():
    net.eval()
    H, W = D.plane_shape(M.shape, o)
    onnx_p = f"{WEB}/pl_unet2d_{o}.onnx"
    dummy = torch.zeros(1, len(D.INPUT_CHANNELS_PLANE), H, W, device=dev)
    torch.onnx.export(net, dummy, onnx_p, opset_version=17, input_names=["x"], output_names=["y"],
                      dynamic_axes={"x": {0: "n"}, "y": {0: "n"}})
    sess = ort.InferenceSession(onnx_p, providers=["CPUExecutionProvider"])
    worst = 0.0
    _tr, _va, _tl, vl = loaders(o)
    for k, (x, _y, _m) in enumerate(vl):
        if k >= 4:
            break
        ref = net(x.to(dev)).float().cpu().numpy()
        got = sess.run(["y"], {"x": x.numpy().astype(np.float32)})[0]
        worst = max(worst, float(np.abs(ref - got).max() * PL_RNG))
    contract = D.surrogate_contract_plane(
        man, M, orient=o, bands=sp.get("train_bands_mhz", man["freqs_mhz"]),
        metrics=dict(plane_rmse_db=round(best_db[o], 3), volume_rmse_db=round(float(np.mean(agg[o])), 3)),
        extra=dict(onnx_parity_db=round(worst, 4), base=BASE, pool=list(orient_pool(o))))
    D.write_surrogate_contract(f"{WEB}/pl_unet2d_{o}.json", contract)
    flag = "OK" if worst <= PARITY_DB else f"FAIL (> {PARITY_DB})"
    print(f"[{o}] wrote {os.path.basename(onnx_p)} + .json  | parity {worst:.4f} dB {flag}")
print("\ndone — three plane models exported.")